#### MFVI -- Mean-Field Variational Inference (Bayes-by-Backprop) auf Qwen-LoRA

Nach Blundell et al. 2015 ("Weight Uncertainty in Neural Networks"). Baut auf `qwen_posterior_utils.py` auf (gleicher `theta_map`, gleiches 128er-Posterior-Subset, gleiche Auswertungspipeline wie in `02_methods_v3.ipynb` fuer MILE) -- fuer einen fairen Methodenvergleich.


In [ ]:
from pathlib import Path

# ============================================================
# CONFIG
# MFVI
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"

BASELINE_SCRIPT = (
    QWEN_REPO
    / "experiments"
    / "ag_news_qwen_lora"
    / "evaluate_saved_baseline.py"
)

RESULT_DIR = (
    MASTER_DIR
    / "method_results"
    / "mfvi_qwen_agnews"
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 2
POSTERIOR_KEY_SEED = 2027   # gleich wie MILE -- gleiches Subset

# ------------------------------------------------------------
# Posterior data (identisch zu 02_methods_v3.ipynb / MILE)
# ------------------------------------------------------------

N_PER_CLASS = 32         # 32 x 4 = 128 examples
SEQ_LEN = 32

# ------------------------------------------------------------
# Prior (identisch zu MILE)
# ------------------------------------------------------------

# War 1.0 (Standard-Default aus dem MILE-Paper). Bei 540672 Dimensionen
# ist das massiv zu breit: eine typische Stichprobe aus N(0, 1.0^2 * I)
# hat eine Norm von ~sqrt(540672)*1.0 ~= 735 -- weit weg von der
# tatsaechlichen MAP-Norm (~13.0). Das MILE-Paper selbst skaliert die
# Prior-Varianz fuer groessere Modelle runter (0.1-0.4 statt 1.0 fuer
# ihre CNN/ATT-Modelle, deutlich kleiner als unser 540k-dim LoRA-Raum).
# Empirisch hergeleitet aus der MAP-Norm: 13.0148 / sqrt(540672) ~= 0.0177.
PRIOR_STD = 0.02

# ------------------------------------------------------------
# MFVI
# ------------------------------------------------------------

N_ELBO_STEPS = 200
LEARNING_RATE = 1e-3

# sigma_init = softplus(INIT_RHO) -- klein, damit das Training nahe
# der schon trainierten MAP-Loesung startet.
INIT_RHO = -5.0

SIGMA_FLOOR = 1e-6   # numerische Untergrenze fuer sigma

N_SAMPLES = 10   # Posterior-Draws fuer die Auswertung (wie bei MILE)

PROGRESS_EVERY = 20

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64

# False (Default)  -> auswerten auf dem 128er-Posterior-Subset
#                      (schneller Dev-/Stabilitaets-Check).
# True              -> auswerten auf dem echten AG-News-Testset
#                      (7600 Beispiele, fuer die finalen Thesis-Zahlen).
EVAL_ON_TEST_SET = False

# ------------------------------------------------------------
# Automatic run name
# ------------------------------------------------------------

N_POSTERIOR_CONFIG = 4 * N_PER_CLASS

RUN_NAME = (
    f"mfvi_balanced{N_POSTERIOR_CONFIG}"
    f"_seq{SEQ_LEN}"
    f"_prior{PRIOR_STD}"
    f"_steps{N_ELBO_STEPS}"
    f"_samples{N_SAMPLES}"
)

print("======================================")
print("MFVI CONFIG")
print("======================================")
print("Run name:          ", RUN_NAME)
print("Examples/class:    ", N_PER_CLASS)
print("Total examples:    ", N_POSTERIOR_CONFIG)
print("Sequence length:   ", SEQ_LEN)
print("Prior std:         ", PRIOR_STD)
print("ELBO steps:        ", N_ELBO_STEPS)
print("Learning rate:     ", LEARNING_RATE)
print("Init rho:          ", INIT_RHO)
print("Posterior samples: ", N_SAMPLES)
print("======================================")


In [ ]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import gc
import copy
import json
import runpy

import numpy as np
import jax
import jax.numpy as jnp
import optax

from jax import random


RESULT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(QWEN_REPO)

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

print("\nPaths:")
print("Qwen repo:", QWEN_REPO.exists())
print("Baseline script:", BASELINE_SCRIPT.exists())
print("Result directory:", RESULT_DIR)

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()


In [ ]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


In [ ]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

test_metrics = baseline_objects["test_metrics"]

print("\nBaseline loaded")
print("Model:", type(env.s_model))

print("\nBaseline test metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {float(value):.6f}")


In [ ]:
# ============================================================
# POSTERIOR SETUP (gleiches Subset, gleicher theta_map wie MILE)
# ============================================================

posterior = qpu.setup_qwen_posterior(
    env=env,
    params=params,
    data=data,
    n_per_class=N_PER_CLASS,
    seq_len=SEQ_LEN,
    posterior_key_seed=POSTERIOR_KEY_SEED,
    prior_std=PRIOR_STD,
)

x_posterior = posterior.x_posterior
y_posterior = posterior.y_posterior
posterior_example_keys = posterior.posterior_example_keys
N_POSTERIOR_EXAMPLES = posterior.n_posterior_examples
subset_indices_host = posterior.subset_indices_host

theta_map = posterior.theta_map
rebuild_full_params = posterior.rebuild_full_params
qwen_subset_nll_sum = posterior.qwen_subset_nll_sum
qwen_log_posterior = posterior.qwen_log_posterior


### ELBO

`nll_fn(theta)` ist dieselbe NLL wie in MILEs `qwen_log_posterior`, nur ohne den Prior-Term -- der Prior geht hier stattdessen analytisch in die Gauss-KL ein (niedrigere Varianz als ein Monte-Carlo-Schaetzer des Priors, Standardpraxis bei Bayes-by-Backprop mit Gauss-Prior + Gauss-Posterior).


In [ ]:
# ============================================================
# ELBO / REPARAMETRIZATION
# ============================================================

D = int(theta_map.shape[0])


def nll_fn(theta):
    candidate_params = rebuild_full_params(theta)
    nll_sum, _ = qwen_subset_nll_sum(
        candidate_params,
        x_posterior,
        y_posterior,
        posterior_example_keys,
    )
    return nll_sum


def gaussian_kl(mu, sigma, prior_std):
    """KL[N(mu, sigma^2) || N(0, prior_std^2)], summed over all dims."""
    return jnp.sum(
        jnp.log(prior_std / sigma)
        + (jnp.square(sigma) + jnp.square(mu)) / (2.0 * prior_std**2)
        - 0.5
    )


def negative_elbo(variational_params, rng_key):
    mu = variational_params["mu"]
    rho = variational_params["rho"]

    sigma = jax.nn.softplus(rho) + SIGMA_FLOOR

    epsilon = random.normal(rng_key, mu.shape, dtype=mu.dtype)
    theta_sample = mu + sigma * epsilon

    nll = nll_fn(theta_sample)
    kl = gaussian_kl(mu, sigma, PRIOR_STD)

    neg_elbo = nll + kl

    return neg_elbo, {"nll": nll, "kl": kl, "sigma_mean": sigma.mean()}


In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

variational_params = {
    "mu": theta_map,
    "rho": jnp.full((D,), INIT_RHO, dtype=jnp.float32),
}

optimizer = optax.adamw(LEARNING_RATE)
opt_state = optimizer.init(variational_params)

grad_fn = jax.value_and_grad(negative_elbo, has_aux=True)


def train_step(variational_params, opt_state, rng_key):
    (loss, aux), grads = grad_fn(variational_params, rng_key)
    updates, opt_state = optimizer.update(grads, opt_state, variational_params)
    variational_params = optax.apply_updates(variational_params, updates)
    return variational_params, opt_state, loss, aux


print("======================================")
print("MFVI TRAINING")
print("======================================")

loss_history = []
nll_history = []
kl_history = []
sigma_mean_history = []

for step in range(N_ELBO_STEPS):
    rng_key, step_key = random.split(rng_key)

    variational_params, opt_state, loss, aux = train_step(
        variational_params, opt_state, step_key,
    )

    loss_host = float(loss)
    nll_host = float(aux["nll"])
    kl_host = float(aux["kl"])
    sigma_mean_host = float(aux["sigma_mean"])

    loss_history.append(loss_host)
    nll_history.append(nll_host)
    kl_history.append(kl_host)
    sigma_mean_history.append(sigma_mean_host)

    finite = np.isfinite(loss_host)

    if step % PROGRESS_EVERY == 0 or step == N_ELBO_STEPS - 1 or not finite:
        print(
            f"step {step:4d}  neg_elbo={loss_host:12.3f}  "
            f"nll={nll_host:12.3f}  kl={kl_host:10.3f}  "
            f"sigma_mean={sigma_mean_host:.6f}"
        )

    if not finite:
        raise FloatingPointError(
            f"MFVI diverged at step {step}: neg_elbo is not finite."
        )

print()
print("MFVI training completed.")
print("Final neg ELBO:", loss_history[-1])
print("Final mean sigma:", sigma_mean_history[-1])


In [ ]:
# ============================================================
# DRAW POSTERIOR SAMPLES FROM TRAINED q
# ============================================================

mu_final = variational_params["mu"]
sigma_final = jax.nn.softplus(variational_params["rho"]) + SIGMA_FLOOR

rng_key, sampling_key = random.split(rng_key)
sampling_keys = random.split(sampling_key, N_SAMPLES)


def draw_one_sample(key):
    epsilon = random.normal(key, mu_final.shape, dtype=mu_final.dtype)
    return mu_final + sigma_final * epsilon


sample_positions = jax.vmap(draw_one_sample)(sampling_keys)
sample_positions = jax.block_until_ready(sample_positions)

samples_host = np.asarray(jax.device_get(sample_positions))

finite_per_sample = np.isfinite(samples_host).all(axis=1)

print("Samples shape:", samples_host.shape)
print("Mean sigma:", float(sigma_final.mean()))
print("Min/Max sigma:", float(sigma_final.min()), float(sigma_final.max()))
print("All samples finite:", finite_per_sample.all())

assert finite_per_sample.all(), "At least one MFVI sample contains NaN or Inf."


In [ ]:
# ============================================================
# BASIC SAMPLE DIAGNOSTICS
# ============================================================

distances_from_map = qpu.compute_distances_from_map(theta_map, sample_positions)


In [ ]:
# ============================================================
# EVALUATION DATA (Posterior-Subset oder echtes Testset)
# ============================================================

if EVAL_ON_TEST_SET:
    x_eval, y_eval = data.get("test")
    print("Evaluating on the full AG News TEST set.")
else:
    x_eval, y_eval = x_posterior, y_posterior
    print("Evaluating on the posterior subset (dev/stability check).")

print("Evaluation examples:", int(y_eval.shape[0]))


In [ ]:
# ============================================================
# POSTERIOR PREDICTIVE PROBABILITIES
# ============================================================

sample_probabilities, mean_probabilities, rng_key = (
    qpu.compute_posterior_predictive_probabilities(
        env=env,
        rebuild_full_params=rebuild_full_params,
        sample_thetas=samples_host,
        x_eval=x_eval,
        y_eval=y_eval,
        rng_key=rng_key,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
)


In [ ]:
# ============================================================
# METRICS
# ============================================================

metrics = qpu.compute_predictive_metrics(
    sample_probabilities=sample_probabilities,
    mean_probabilities=mean_probabilities,
    y_true=y_eval,
)


In [ ]:
# ============================================================
# EXPECTED CALIBRATION ERROR
# ============================================================

ece = qpu.multiclass_ece(mean_probabilities, y_eval, n_bins=15)
print("ECE:", float(ece))


In [ ]:
# ============================================================
# LOG POSTERIOR VALUES FOR ALL SAMPLES
# ============================================================

sample_log_posteriors_host = qpu.compute_sample_log_posteriors(
    qwen_log_posterior,
    sample_positions,
)


In [ ]:
# ============================================================
# SAVE RUN
# ============================================================

summary, metadata_arrays = qpu.summarize_metrics(metrics)

summary["ece"] = float(ece)

summary.update({
    "run_name": RUN_NAME,
    "method": "MFVI",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",

    "seed": SEED,
    "posterior_key_seed": POSTERIOR_KEY_SEED,

    "n_per_class": N_PER_CLASS,
    "n_posterior_examples": N_POSTERIOR_EXAMPLES,
    "sequence_length": SEQ_LEN,

    "eval_on_test_set": EVAL_ON_TEST_SET,
    "n_eval_examples": int(y_eval.shape[0]),

    "prior_std": PRIOR_STD,

    "n_elbo_steps": N_ELBO_STEPS,
    "learning_rate": LEARNING_RATE,
    "init_rho": INIT_RHO,
    "n_samples": N_SAMPLES,

    "final_neg_elbo": loss_history[-1],
    "final_nll": nll_history[-1],
    "final_kl": kl_history[-1],
    "final_sigma_mean": sigma_mean_history[-1],
    "final_sigma_min": float(sigma_final.min()),
    "final_sigma_max": float(sigma_final.max()),

    "all_samples_finite": bool(np.isfinite(samples_host).all()),
    "n_parameter_dimensions": int(samples_host.shape[-1]),

    "minimum_distance_from_map": float(distances_from_map.min()),
    "mean_distance_from_map": float(distances_from_map.mean()),
    "maximum_distance_from_map": float(distances_from_map.max()),

    "minimum_log_posterior": float(sample_log_posteriors_host.min()),
    "mean_log_posterior": float(sample_log_posteriors_host.mean()),
    "maximum_log_posterior": float(sample_log_posteriors_host.max()),
})

metadata_arrays.update({
    "subset_indices": np.asarray(subset_indices_host),
    "labels": np.asarray(jax.device_get(y_eval)),
    "distances_from_map": np.asarray(distances_from_map),
    "log_posteriors": sample_log_posteriors_host,
    "loss_history": np.asarray(loss_history),
    "nll_history": np.asarray(nll_history),
    "kl_history": np.asarray(kl_history),
    "sigma_mean_history": np.asarray(sigma_mean_history),
    "final_sigma": np.asarray(jax.device_get(sigma_final)),
})

paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=RUN_NAME,
    sample_positions=sample_positions,
    sample_probabilities=sample_probabilities,
    metadata_arrays=metadata_arrays,
    summary=summary,
)

print()
print("Results:")
print("Accuracy:           ", summary["accuracy"])
print("LPPD:               ", summary["lppd"])
print("NLL:                ", summary["posterior_predictive_nll"])
print("Brier Score:        ", summary["brier_score"])
print("ECE:                ", summary["ece"])
print("Mean pred. entropy: ", summary["mean_predictive_entropy"])
print("Mean exp. entropy:  ", summary["mean_expected_entropy"])
print("Mean MI:            ", summary["mean_mutual_information"])
